# Consolidated Budget Pack (S8)

Brings the Phase 1 interventions together against the GBP 1,000,000 "Make It Visible" package,
with **everything in**:

- Tactical corridor (city-core amber spine)
- Tiered digital wayfinders + gateway hub
- **Gateway pavilion - WOODEN, REVERSIBLE (demountable / meanwhile-use)**
- **Dartmouth crossing - PHASE-1 tactical enhancement** (full single-stage/Toucan upgrade is a later phase)

**Accuracy approach:** quantities are from the analysis (Level 3). Unit costs are INDICATIVE
benchmark assumptions (Level 1) pending a quantity-surveyor costing, so every line is a
**low / central / high range**. Sponsorship is a separate potential offset. No funding-facing
cost claim is made.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
PHASE1_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT / "phase1_spinelens_ai"
sys.path.insert(0, str(PHASE1_ROOT / "src"))
from spinelens.models import budget as bg
from spinelens.gate0b import utc_now_iso

DATA = PHASE1_ROOT / "data"; TABLES = PHASE1_ROOT / "outputs" / "tables"
FIG_DIR = PHASE1_ROOT / "outputs" / "reports" / "budget_media"; REPORTS = PHASE1_ROOT / "outputs" / "reports"
for d in (FIG_DIR, TABLES, REPORTS): d.mkdir(parents=True, exist_ok=True)
ENVELOPE = 1_000_000
DESIGN_PCT, CONTINGENCY_PCT = 13, 17

# Quantities from the analysis (Level 3)
seg = pd.read_csv(TABLES / "tactical_corridor_segment_scores.csv")
trunk_m = float(seg.loc[seg.multiplicity >= 2, "length_m"].sum())
spur_m = float(seg.loc[seg.multiplicity == 1, "length_m"].sum())
wc = pd.read_csv(TABLES / "wayfinder_content_pack.csv")
n_totem = int((wc["tier"] == "tier2_totem").sum())
n_light = int(((wc["tier"] == "tier3_marker") & (wc["content_role"] == "light_marker")).sum())
n_nech = int(((wc["tier"] == "tier3_marker") & (wc["content_role"] == "dwell_no_pavilion")).sum())
SPUR_FRAC = {"low": 0.40, "central": 0.55, "high": 0.70}
print(f"corridor trunk {trunk_m:.0f} m, spur {spur_m:.0f} m | totems {n_totem}, light {n_light}, Nechells marker {n_nech}")

## Phase 1 cost lines (everything in)

In [ ]:
lines = []
def add(line, category, qty_evidence, cost):
    lines.append({"line": line, "category": category, "phase": "phase1",
                  "quantity_evidence": qty_evidence, "cost_evidence": "L1 indicative", **cost})

add("Tactical corridor - shared trunk", "corridor", "L3", bg.line_total(trunk_m, 100, 150, 250))
add("Tactical corridor - spurs (treated portion)", "corridor", "L3",
    {"low": round(spur_m*SPUR_FRAC["low"]*40), "central": round(spur_m*SPUR_FRAC["central"]*80),
     "high": round(spur_m*SPUR_FRAC["high"]*150)})
add("Interactive wayfinder totems (Tier 2)", "wayfinders", "L3", bg.line_total(n_totem, 8000, 16000, 25000))
add("Light markers (Tier 3, city-core)", "wayfinders", "L3", bg.line_total(n_light, 2000, 4000, 6000))
add("Nechells boosted marker (no-pavilion)", "wayfinders", "L3", bg.line_total(n_nech, 6000, 10000, 15000))
add("Gateway hub digital fit-out (Tier 1)", "wayfinders", "L3", bg.line_total(1, 25000, 45000, 75000))
add("Gateway pavilion - WOODEN, REVERSIBLE (demountable)", "pavilion", "L2", bg.line_total(1, 80000, 150000, 280000))
add("Dartmouth crossing - Phase-1 tactical enhancement", "crossing", "L3", bg.line_total(1, 40000, 90000, 150000))

budget = pd.DataFrame(lines)
budget.to_csv(TABLES / "phase1_budget_lines.csv", index=False)
display(budget[["line", "category", "low", "central", "high", "quantity_evidence", "cost_evidence"]])

## Phase 1 total vs the GBP 1,000,000 envelope (everything in)

In [ ]:
construction = bg.sum_costs(lines)
with_fees = bg.add_percentage(construction, DESIGN_PCT)
with_cont = bg.add_percentage(with_fees, CONTINGENCY_PCT)
activation = bg.line_total(1, 20000, 45000, 80000)
total = bg.sum_costs([with_cont, activation])
sponsorship = bg.line_total(1, 10000, 25000, 40000)
net = bg.apply_offset(total, sponsorship)
ec = bg.envelope_check(net, ENVELOPE)

print("Construction (everything in) central: GBP {:,}".format(construction["central"]))
print("Total incl. {}% design + {}% contingency + activation:".format(DESIGN_PCT, CONTINGENCY_PCT))
print("  {:,} - {:,} - {:,}".format(total["low"], total["central"], total["high"]))
print("Net of potential civic-partner sponsorship:")
print("  {:,} - {:,} - {:,}".format(net["low"], net["central"], net["high"]))
print("Envelope check vs GBP 1,000,000:", ec)

In [ ]:
# Crossing allowance: how much could the crossing grow and still fit GBP 1,000,000?
soft_mult = (1 + DESIGN_PCT/100) * (1 + CONTINGENCY_PCT/100)
base_no_crossing = bg.sum_costs([l for l in lines if l["category"] != "crossing"])
crossing_line = next(l for l in lines if l["category"] == "crossing")
for band in ["central", "high"]:
    allowance = (ENVELOPE + sponsorship[band] - activation[band]) / soft_mult - base_no_crossing[band]
    print(f"Crossing allowance at {band}: ~GBP{round(allowance):,} "
          f"(Phase-1 crossing {band} GBP{crossing_line[band]:,})")
print("\nLater-phase reference (NOT in the Phase 1 budget):")
print("  Full crossing upgrade (single-stage signal + Toucan, Gate 0C): GBP 200,000 - 375,000 - 550,000")
print("  -> pursued as separate highways / active-travel capital in a later phase.")

## Visuals

In [ ]:
cat = budget.groupby("category")[["low", "central", "high"]].sum().sort_values("central")
fig, ax = plt.subplots(figsize=(10, 5))
err = np.vstack([cat["central"]-cat["low"], cat["high"]-cat["central"]])
ax.barh(cat.index, cat["central"]/1000, xerr=err/1000, color="#1a73e8", ecolor="#5f6368", capsize=4)
ax.set_xlabel("GBP thousands (central, low-high range)")
ax.set_title("Phase 1 cost by category - everything in (before soft costs)")
for i, v in enumerate(cat["central"]):
    ax.text(v/1000, i, f" {v/1000:.0f}k", va="center", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR / "figT_cost_by_category.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
bands = ["low", "central", "high"]
ax.bar(bands, [net[b]/1000 for b in bands], color=["#34a853", "#1a73e8", "#fbbc04"])
ax.axhline(ENVELOPE/1000, color="#d93025", ls="--", lw=1.5, label="GBP 1,000,000 envelope")
for i, b in enumerate(bands):
    ax.text(i, net[b]/1000, f" {net[b]/1000:.0f}k", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("GBP thousands"); ax.set_title("Phase 1 'everything in' total (net of sponsorship) vs envelope")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "figU_phase1_vs_envelope.png", dpi=130, bbox_inches="tight"); plt.show()

## Note

In [ ]:
ts = utc_now_iso()
def g(d): return f"GBP {d['low']:,} - {d['central']:,} - {d['high']:,}"
allow_c = (ENVELOPE + sponsorship["central"] - activation["central"]) / soft_mult - base_no_crossing["central"]
lines_note = [
    "# Phase 1 Budget Pack Note (S8)",
    "", f"Generated: {ts}. Everything in. Indicative ranges; unit costs Level 1 pending QS. No funding claim.",
    "", "## Decision reflected", "",
    "- Pavilion is **wooden and reversible (demountable / meanwhile-use)** - cheaper and more",
    "  deliverable: a low timber structure eases helipad height/safeguarding, and a temporary",
    "  reversible structure needs a licence/meanwhile-use agreement, not permanent land acquisition",
    "  (the ownership risk drops from acquisition to licence).",
    "- Crossing starts as a **Phase-1 tactical enhancement** of the existing controlled crossing;",
    "  the full single-stage/Toucan upgrade is a **later-phase** capital improvement.",
    "", "## Phase 1 total (everything in)", "",
    f"- Construction central: GBP {construction['central']:,}.",
    f"- Total (incl. {DESIGN_PCT}% design + {CONTINGENCY_PCT}% contingency + activation): {g(total)}.",
    f"- Net of potential civic-partner sponsorship: {g(net)}.",
    f"- Fit vs GBP 1,000,000: central {'fits' if ec['fits_central'] else 'exceeds'} "
    f"(headroom GBP {ec['headroom_central']:,}); high band {'fits' if ec['fits_high'] else 'exceeds'}.",
    "", "## Flex on the crossing", "",
    f"- At central costs the crossing could grow to ~GBP {round(allow_c):,} and still fit GBP 1,000,000,",
    "  so the Phase-1 crossing can be scaled up within budget if a more ambitious first step is wanted.",
    "", "## Honest caveat", "",
    "- The HIGH cost band exceeds GBP 1,000,000; the central headroom is the buffer against cost risk.",
    "- All unit costs are Level 1 indicative; a quantity-surveyor costing is the priority validation.",
    "", "## Later-phase reference (not in Phase 1 budget)", "",
    "- Full crossing upgrade (single-stage signal + Toucan): GBP 200,000 - 375,000 - 550,000, as separate",
    "  highways / active-travel capital.",
    "- Permanent pavilion, if the reversible pilot proves the case.",
]
(REPORTS / "budget_pack_note.md").write_text("\n".join(lines_note), encoding="utf-8")
print("\n".join(lines_note[:20]))

## What this unlocks

A Phase 1 budget with **everything in** that fits GBP 1,000,000 at central with headroom: a
reversible timber pavilion (cheaper and more deliverable), a phased crossing (tactical now,
full upgrade later), plus the corridor, wayfinders and hub. Ready for a QS to firm the unit
costs and collapse the range.